# Task 160: enterprise_pta — Inverse Problem Tutorial

## 1. Problem Background and Mathematical Description

**Task**: Pulsar timing array gravitational wave detection using enterprise

**Paper**: ENTERPRISE: Enhanced Numerical Toolbox Enabling a Robust PulsaR Inference SuitE
**Repository**: https://github.com/nanograv/enterprise

### Physical Background

Spectroscopic measurements encode material properties in spectral features. Fitting extracts quantitative information.

### Forward Model

```
S(v) = sum_k A_k P_k(v; theta_k) + B(v)
```

### Inverse Problem

```
Nonlinear least-squares fitting or matrix decomposition (MCR-ALS, NMF)
```

### Performance Metrics
- **PSNR**: N/A (parameter estimation)
- **SSIM**: N/A
- **Evaluation**: PTA Bayesian inference (parameter RE values, power spectrum CC)

### Mathematical Formulation

The general inverse problem seeks to recover $\mathbf{x}$ from indirect measurements:

$$\mathbf{y} = \mathcal{A}(\mathbf{x}) + \boldsymbol{\eta}$$

**Regularized solution**:
$$\hat{\mathbf{x}} = \arg\min_{\mathbf{x}} \frac{1}{2}\|\mathcal{A}(\mathbf{x}) - \mathbf{y}\|_2^2 + \lambda \mathcal{R}(\mathbf{x})$$

The regularization parameter $\lambda$ balances data fidelity against prior constraints, controlling the bias-variance trade-off.


## 2. Environment Setup and Library Imports

Import numerical, visualization, and domain-specific libraries needed for this tutorial.

In [ ]:
#!/usr/bin/env python
"""
Pulsar Timing Array (PTA) Bayesian Inference
=============================================
Simulates a pulsar timing array with 5 pulsars, each having:
  - White noise
  - Red noise (power-law spectrum)
  - Gravitational wave background (GWB, Hellings-Downs correlated)

Uses emcee MCMC to recover 3 key parameters:
  - log10_A_gw   : amplitude of GW background
  - log10_A_red  : amplitude of intrinsic red noise
  - gamma_red    : spectral index of red noise

Outputs metrics, visualizations, and numpy arrays for downstream evaluation.
"""

import os
import json
import numpy as np
import matplotlib

## 3. Utility Functions

Helper functions providing mathematical building blocks:
`hellings_downs`, `make_hd_matrix`, `powerlaw_psd`, `fourier_design_matrix`, `log_likelihood`, `log_prior`, `log_posterior`, `main`

In [ ]:
# ── Helper: Hellings-Downs correlation ───────────────────────────────────────
def hellings_downs(theta):
    """Hellings-Downs overlap reduction function."""
    x = (1.0 - np.cos(theta)) / 2.0
    if x < 1e-10:
        return 1.0
    hd = 1.5 * x * np.log(x) - 0.25 * x + 0.5
    return hd

def make_hd_matrix(positions):
    """Build the N_psr x N_psr Hellings-Downs correlation matrix."""
    n = len(positions)
    hd = np.eye(n)
    for i in range(n):
        for j in range(i + 1, n):
            cos_theta = np.dot(positions[i], positions[j])
            cos_theta = np.clip(cos_theta, -1.0, 1.0)
            theta = np.arccos(cos_theta)
            val = hellings_downs(theta)
            hd[i, j] = val
            hd[j, i] = val
    return hd

# ── Helper: power-law PSD ────────────────────────────────────────────────────
def powerlaw_psd(freqs, log10_A, gamma):
    """Power-law power spectral density: S(f) = A^2/(12*pi^2) * (f/f_yr)^(-gamma) * f_yr^(-3)."""
    A = 10.0 ** log10_A
    f_yr = 1.0 / (365.25 * 86400.0)
    return (A ** 2 / (12.0 * np.pi ** 2)) * (freqs / f_yr) ** (-gamma) * f_yr ** (-3)

# ── Generate Fourier design matrix ───────────────────────────────────────────
def fourier_design_matrix(toas, n_freq, T):
    """Create the Fourier design matrix F  (N_toa x 2*n_freq)."""
    N = len(toas)
    F = np.zeros((N, 2 * n_freq))
    freqs = np.arange(1, n_freq + 1) / T
    for i, f in enumerate(freqs):
        F[:, 2 * i] = np.sin(2.0 * np.pi * f * toas)
        F[:, 2 * i + 1] = np.cos(2.0 * np.pi * f * toas)
    return F, freqs

# ── Log-likelihood ────────────────────────────────────────────────────────────
def log_likelihood(params, toas_all, residuals_all, F_all, freqs, hd_matrix):
    """Marginalised log-likelihood for PTA (Fourier-domain)."""
    log10_A_gw, log10_A_red, gamma_red = params

    psd_red = powerlaw_psd(freqs, log10_A_red, gamma_red)
    psd_gw = powerlaw_psd(freqs, log10_A_gw, 13.0 / 3.0)

    logL = 0.0
    sigma_wn_sq = (TRUE_SIGMA_WN * TRUE_EFAC) ** 2

    for p in range(N_PULSARS):
        F = F_all[p]
        r = residuals_all[p]
        N_inv = np.eye(N_TOA) / sigma_wn_sq

        # Per-pulsar phi: red + GW (diagonal part for this pulsar)
        phi_diag = np.zeros(2 * N_FREQ)
        for k in range(N_FREQ):
            phi_val = (psd_red[k] + hd_matrix[p, p] * psd_gw[k]) * T_SPAN
            phi_diag[2 * k] = phi_val
            phi_diag[2 * k + 1] = phi_val

        # Woodbury: (N + F phi F^T)^{-1} via Woodbury identity
        # Sigma = F diag(phi) F^T + N
        # log|Sigma| and r^T Sigma^{-1} r via Woodbury
        phi_inv = np.where(phi_diag > 0, 1.0 / phi_diag, 1e30)
        FNr = F.T @ (r / sigma_wn_sq)
        FNF = F.T @ F / sigma_wn_sq
        Sigma_a = FNF + np.diag(phi_inv)

        try:
            cf = cho_factor(Sigma_a)
            Sigma_a_inv_FNr = cho_solve(cf, FNr)
            log_det_Sigma_a = 2.0 * np.sum(np.log(np.diag(cf[0])))
        except np.linalg.LinAlgError:
            return -1e100

        # log-likelihood contribution
        rNr = np.sum(r ** 2) / sigma_wn_sq
        logL_p = -0.5 * rNr + 0.5 * FNr @ Sigma_a_inv_FNr
        logL_p -= 0.5 * log_det_Sigma_a
        logL_p -= 0.5 * np.sum(np.log(phi_diag[phi_diag > 0]))

        logL += logL_p

    return logL

# ── Prior ─────────────────────────────────────────────────────────────────────
def log_prior(params):
    """Uniform priors on parameters."""
    log10_A_gw, log10_A_red, gamma_red = params
    if not (-18.0 < log10_A_gw < -11.0):
        return -np.inf
    if not (-18.0 < log10_A_red < -11.0):
        return -np.inf
    if not (0.0 < gamma_red < 10.0):
        return -np.inf
    return 0.0

def log_posterior(params, toas_all, residuals_all, F_all, freqs, hd_matrix):
    """Log-posterior = log-prior + log-likelihood."""
    lp = log_prior(params)
    if not np.isfinite(lp):
        return -np.inf
    ll = log_likelihood(params, toas_all, residuals_all, F_all, freqs, hd_matrix)
    if not np.isfinite(ll):
        return -np.inf
    return lp + ll

# ── Main ──────────────────────────────────────────────────────────────────────
def main():
    print("=" * 60)
    print("PTA Bayesian Inference — Enterprise-style")
    print("=" * 60)

    # 1. Simulate data
    print("\n[1/4] Simulating PTA data ...")
    toas_all, residuals_all, F_all, freqs, hd_matrix, positions = simulate_pta()
    print(f"  → {N_PULSARS} pulsars, {N_TOA} TOAs each, {T_SPAN_YR} yr span")

    # 2. Run MCMC
    print(f"\n[2/4] Running MCMC ({N_WALKERS} walkers × {N_STEPS} steps) ...")
    true_params = np.array([TRUE_LOG10_A_GW, TRUE_LOG10_A_RED, TRUE_GAMMA_RED])
    ndim = 3

    # Initialise walkers near truth (with scatter)
    p0 = true_params + 0.3 * np.random.randn(N_WALKERS, ndim)
    # Clip to prior bounds
    p0[:, 0] = np.clip(p0[:, 0], -17.9, -11.1)
    p0[:, 1] = np.clip(p0[:, 1], -17.9, -11.1)
    p0[:, 2] = np.clip(p0[:, 2], 0.1, 9.9)

    sampler = emcee.EnsembleSampler(
        N_WALKERS, ndim, log_posterior,
        args=(toas_all, residuals_all, F_all, freqs, hd_matrix)
    )
    sampler.run_mcmc(p0, N_STEPS, progress=True)
    print("  → MCMC complete")

    # 3. Analyse chains
    print("\n[3/4] Analysing posterior samples ...")
    samples = sampler.get_chain(discard=N_BURN, flat=True)
    param_names = ["log10_A_gw", "log10_A_red", "gamma_red"]
    medians = np.median(samples, axis=0)
    stds = np.std(samples, axis=0)

    print(f"\n  {'Parameter':<15s} {'True':>10s} {'Median':>10s} {'Std':>10s}")
    print("  " + "-" * 50)
    for i, name in enumerate(param_names):
        print(f"  {name:<15s} {true_params[i]:10.3f} {medians[i]:10.3f} {stds[i]:10.3f}")

    # Relative errors
    re_values = {}
    for i, name in enumerate(param_names):
        if abs(true_params[i]) > 1e-10:
            re = abs(medians[i] - true_params[i]) / abs(true_params[i])
        else:
            re = abs(medians[i] - true_params[i])
        re_values[name] = float(re)

    # Compute power spectrum comparison (red noise PSD)
    psd_true = powerlaw_psd(freqs, TRUE_LOG10_A_RED, TRUE_GAMMA_RED)
    psd_recon = powerlaw_psd(freqs, medians[1], medians[2])
    # Cross-correlation of log-PSDs
    log_psd_true = np.log10(psd_true + 1e-100)
    log_psd_recon = np.log10(psd_recon + 1e-100)
    cc_num = np.sum((log_psd_true - log_psd_true.mean()) *
                     (log_psd_recon - log_psd_recon.mean()))
    cc_den = np.sqrt(np.sum((log_psd_true - log_psd_true.mean()) ** 2) *
                     np.sum((log_psd_recon - log_psd_recon.mean()) ** 2))
    psd_cc = float(cc_num / (cc_den + 1e-30))

    # GW background PSD comparison
    psd_gw_true = powerlaw_psd(freqs, TRUE_LOG10_A_GW, 13.0 / 3.0)
    psd_gw_recon = powerlaw_psd(freqs, medians[0], 13.0 / 3.0)
    log_gw_true = np.log10(psd_gw_true + 1e-100)
    log_gw_recon = np.log10(psd_gw_recon + 1e-100)
    cc_gw_num = np.sum((log_gw_true - log_gw_true.mean()) *
                        (log_gw_recon - log_gw_recon.mean()))
    cc_gw_den = np.sqrt(np.sum((log_gw_true - log_gw_true.mean()) ** 2) *
                        np.sum((log_gw_recon - log_gw_recon.mean()) ** 2))
    psd_gw_cc = float(cc_gw_num / (cc_gw_den + 1e-30))

    mean_re = float(np.mean(list(re_values.values())))

    print(f"\n  Mean relative error: {mean_re:.4f}")
    print(f"  Red noise PSD CC:   {psd_cc:.4f}")
    print(f"  GW PSD CC:          {psd_gw_cc:.4f}")

    # 4. Save outputs
    print("\n[4/4] Saving results ...")

    # Metrics
    metrics = {
        "log10_A_gw_true": TRUE_LOG10_A_GW,
        "log10_A_gw_recovered": float(medians[0]),
        "log10_A_gw_std": float(stds[0]),
        "log10_A_gw_RE": re_values["log10_A_gw"],
        "log10_A_red_true": TRUE_LOG10_A_RED,
        "log10_A_red_recovered": float(medians[1]),
        "log10_A_red_std": float(stds[1]),
        "log10_A_red_RE": re_values["log10_A_red"],
        "gamma_red_true": TRUE_GAMMA_RED,
        "gamma_red_recovered": float(medians[2]),
        "gamma_red_std": float(stds[2]),
        "gamma_red_RE": re_values["gamma_red"],
        "mean_parameter_RE": mean_re,
        "red_noise_PSD_CC": psd_cc,
        "GW_PSD_CC": psd_gw_cc,
        "n_pulsars": N_PULSARS,
        "n_toa": N_TOA,
        "n_walkers": N_WALKERS,
        "n_steps": N_STEPS,
        "n_burn": N_BURN,
    }
    with open(os.path.join(RESULTS_DIR, "metrics.json"), "w") as f:
        json.dump(metrics, f, indent=2)
    print("  → metrics.json saved")

    # Ground truth & reconstruction arrays
    gt_dict = {
        "true_params": true_params,
        "freqs": freqs,
        "psd_red_true": psd_true,
        "psd_gw_true": psd_gw_true,
        "residuals": [r for r in residuals_all],
        "hd_matrix": hd_matrix,
    }
    np.save(os.path.join(RESULTS_DIR, "ground_truth.npy"), gt_dict,
            allow_pickle=True)

    recon_dict = {
        "recovered_params": medians,
        "param_stds": stds,
        "psd_red_recon": psd_recon,
        "psd_gw_recon": psd_gw_recon,
        "samples": samples,
    }
    np.save(os.path.join(RESULTS_DIR, "reconstruction.npy"), recon_dict,
            allow_pickle=True)
    print("  → ground_truth.npy, reconstruction.npy saved")

    # ── Visualization ─────────────────────────────────────────────────────
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))

    # (0,0) Trace plots
    chain = sampler.get_chain()
    for i, name in enumerate(param_names):
        ax = axes[0, 0] if i == 0 else (axes[0, 1] if i == 1 else axes[0, 2])
        for w in range(N_WALKERS):
            ax.plot(chain[:, w, i], alpha=0.3, lw=0.5)
        ax.axhline(true_params[i], color='r', lw=2, label='Truth')
        ax.axhline(medians[i], color='blue', ls='--', lw=1.5, label='Median')
        ax.set_xlabel('Step')
        ax.set_ylabel(name)
        ax.set_title(f'Trace: {name}')
        ax.legend(fontsize=8)

    # (1,0) Red noise PSD
    ax = axes[1, 0]
    ax.loglog(freqs * 365.25 * 86400, psd_true, 'r-', lw=2, label='True red noise')
    ax.loglog(freqs * 365.25 * 86400, psd_recon, 'b--', lw=2, label='Recovered')
    ax.set_xlabel('Frequency (1/yr)')
    ax.set_ylabel('PSD (s²/Hz)')
    ax.set_title(f'Red Noise PSD (CC={psd_cc:.3f})')
    ax.legend()
    ax.grid(True, alpha=0.3)

    # (1,1) GW PSD
    ax = axes[1, 1]
    ax.loglog(freqs * 365.25 * 86400, psd_gw_true, 'r-', lw=2, label='True GWB')
    ax.loglog(freqs * 365.25 * 86400, psd_gw_recon, 'b--', lw=2, label='Recovered')
    ax.set_xlabel('Frequency (1/yr)')
    ax.set_ylabel('PSD (s²/Hz)')
    ax.set_title(f'GW Background PSD (CC={psd_gw_cc:.3f})')
    ax.legend()
    ax.grid(True, alpha=0.3)

    # (1,2) Corner-like: 2D posterior (A_gw vs A_red)
    ax = axes[1, 2]
    ax.scatter(samples[:, 0], samples[:, 1], s=1, alpha=0.2, c='steelblue')
    ax.axvline(TRUE_LOG10_A_GW, color='r', lw=1.5, label='True')
    ax.axhline(TRUE_LOG10_A_RED, color='r', lw=1.5)
    ax.scatter([medians[0]], [medians[1]], c='blue', marker='x', s=100,
               zorder=5, label='Median')
    ax.set_xlabel('log10_A_gw')
    ax.set_ylabel('log10_A_red')
    ax.set_title('Posterior: A_gw vs A_red')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    fig_path = os.path.join(RESULTS_DIR, "reconstruction_result.png")
    plt.savefig(fig_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"  → {fig_path} saved")

    print("\n" + "=" * 60)
    print("DONE — PTA Bayesian inference complete")
    print(f"  Mean param RE = {mean_re:.4f}")
    print(f"  Red PSD CC    = {psd_cc:.4f}")
    print(f"  GW PSD CC     = {psd_gw_cc:.4f}")
    print("=" * 60)

## 4. Forward Model

Define the forward operator mapping true model to observations.

```
S(v) = sum_k A_k P_k(v; theta_k) + B(v)
```

Functions: `simulate_pta`

In [ ]:
# ── Simulate PTA data ────────────────────────────────────────────────────────
def simulate_pta():
    """Simulate timing residuals for N_PULSARS pulsars."""
    # Random sky positions (unit vectors)
    positions = []
    for _ in range(N_PULSARS):
        phi = np.random.uniform(0, 2 * np.pi)
        cos_theta = np.random.uniform(-1, 1)
        sin_theta = np.sqrt(1 - cos_theta ** 2)
        positions.append(np.array([sin_theta * np.cos(phi),
                                   sin_theta * np.sin(phi),
                                   cos_theta]))

    hd_matrix = make_hd_matrix(positions)
    freqs = np.arange(1, N_FREQ + 1) / T_SPAN

    # Red noise PSD per frequency
    psd_red = powerlaw_psd(freqs, TRUE_LOG10_A_RED, TRUE_GAMMA_RED)
    # GW PSD per frequency (GWB spectral index = 13/3 for circular binaries)
    psd_gw = powerlaw_psd(freqs, TRUE_LOG10_A_GW, 13.0 / 3.0)

    toas_all = []
    residuals_all = []
    F_all = []

    for p in range(N_PULSARS):
        # Uniform TOAs
        toas = np.linspace(0, T_SPAN, N_TOA)
        toas_all.append(toas)

        F, _ = fourier_design_matrix(toas, N_FREQ, T_SPAN)
        F_all.append(F)

    # Generate correlated GW Fourier coefficients across pulsars
    # For each frequency, draw correlated coefficients using HD matrix
    gw_coeffs = np.zeros((N_PULSARS, 2 * N_FREQ))
    for k in range(N_FREQ):
        amplitude = np.sqrt(psd_gw[k] * T_SPAN)
        # Cholesky of HD matrix for correlation
        L = np.linalg.cholesky(hd_matrix + 1e-10 * np.eye(N_PULSARS))
        for c in range(2):  # sin and cos
            uncorr = np.random.randn(N_PULSARS)
            corr = L @ uncorr * amplitude
            gw_coeffs[:, 2 * k + c] = corr

    for p in range(N_PULSARS):
        F = F_all[p]
        toas = toas_all[p]

        # Red noise (independent per pulsar)
        red_coeffs = np.zeros(2 * N_FREQ)
        for k in range(N_FREQ):
            amplitude = np.sqrt(psd_red[k] * T_SPAN)
            red_coeffs[2 * k] = np.random.randn() * amplitude
            red_coeffs[2 * k + 1] = np.random.randn() * amplitude

        # Total signal
        signal = F @ (red_coeffs + gw_coeffs[p])

        # White noise
        wn = np.random.randn(N_TOA) * TRUE_SIGMA_WN * TRUE_EFAC

        residuals = signal + wn
        residuals_all.append(residuals)

    return toas_all, residuals_all, F_all, freqs, hd_matrix, positions

## 5. Execution Pipeline

Now we run the complete inverse problem pipeline: data generation, forward modeling, inversion, and analysis.

### Processing Step

Intermediate processing step in the pipeline.

In [ ]:
print("=" * 60)
print("PTA Bayesian Inference — Enterprise-style")
print("=" * 60)

### 1. Simulate data

Apply the forward operator to ground truth to simulate realistic measurements, modeling the physical acquisition process including noise.

In [ ]:
# 1. Simulate data
print("\n[1/4] Simulating PTA data ...")
toas_all, residuals_all, F_all, freqs, hd_matrix, positions = simulate_pta()
print(f"  → {N_PULSARS} pulsars, {N_TOA} TOAs each, {T_SPAN_YR} yr span")

### 2. Run MCMC

Intermediate processing step in the pipeline.

In [ ]:
# 2. Run MCMC
print(f"\n[2/4] Running MCMC ({N_WALKERS} walkers × {N_STEPS} steps) ...")
true_params = np.array([TRUE_LOG10_A_GW, TRUE_LOG10_A_RED, TRUE_GAMMA_RED])
ndim = 3

# Initialise walkers near truth (with scatter)
p0 = true_params + 0.3 * np.random.randn(N_WALKERS, ndim)
# Clip to prior bounds
p0[:, 0] = np.clip(p0[:, 0], -17.9, -11.1)
p0[:, 1] = np.clip(p0[:, 1], -17.9, -11.1)
p0[:, 2] = np.clip(p0[:, 2], 0.1, 9.9)

sampler = emcee.EnsembleSampler(
    N_WALKERS, ndim, log_posterior,
    args=(toas_all, residuals_all, F_all, freqs, hd_matrix)
)
sampler.run_mcmc(p0, N_STEPS, progress=True)
print("  → MCMC complete")

### 3. Analyse chains

Set up experiment parameters: grid sizes, noise levels, regularization weights, algorithm settings.

In [ ]:
# 3. Analyse chains
print("\n[3/4] Analysing posterior samples ...")
samples = sampler.get_chain(discard=N_BURN, flat=True)
param_names = ["log10_A_gw", "log10_A_red", "gamma_red"]
medians = np.median(samples, axis=0)
stds = np.std(samples, axis=0)

print(f"\n  {'Parameter':<15s} {'True':>10s} {'Median':>10s} {'Std':>10s}")
print("  " + "-" * 50)
for i, name in enumerate(param_names):
    print(f"  {name:<15s} {true_params[i]:10.3f} {medians[i]:10.3f} {stds[i]:10.3f}")

# Relative errors
re_values = {}
for i, name in enumerate(param_names):
    if abs(true_params[i]) > 1e-10:
        re = abs(medians[i] - true_params[i]) / abs(true_params[i])
    else:
        re = abs(medians[i] - true_params[i])
    re_values[name] = float(re)

# Compute power spectrum comparison (red noise PSD)
psd_true = powerlaw_psd(freqs, TRUE_LOG10_A_RED, TRUE_GAMMA_RED)
psd_recon = powerlaw_psd(freqs, medians[1], medians[2])
# Cross-correlation of log-PSDs
log_psd_true = np.log10(psd_true + 1e-100)
log_psd_recon = np.log10(psd_recon + 1e-100)
cc_num = np.sum((log_psd_true - log_psd_true.mean()) *
                 (log_psd_recon - log_psd_recon.mean()))
cc_den = np.sqrt(np.sum((log_psd_true - log_psd_true.mean()) ** 2) *
                 np.sum((log_psd_recon - log_psd_recon.mean()) ** 2))
psd_cc = float(cc_num / (cc_den + 1e-30))

# GW background PSD comparison
psd_gw_true = powerlaw_psd(freqs, TRUE_LOG10_A_GW, 13.0 / 3.0)
psd_gw_recon = powerlaw_psd(freqs, medians[0], 13.0 / 3.0)
log_gw_true = np.log10(psd_gw_true + 1e-100)
log_gw_recon = np.log10(psd_gw_recon + 1e-100)
cc_gw_num = np.sum((log_gw_true - log_gw_true.mean()) *
                    (log_gw_recon - log_gw_recon.mean()))
cc_gw_den = np.sqrt(np.sum((log_gw_true - log_gw_true.mean()) ** 2) *
                    np.sum((log_gw_recon - log_gw_recon.mean()) ** 2))
psd_gw_cc = float(cc_gw_num / (cc_gw_den + 1e-30))

mean_re = float(np.mean(list(re_values.values())))

print(f"\n  Mean relative error: {mean_re:.4f}")
print(f"  Red noise PSD CC:   {psd_cc:.4f}")
print(f"  GW PSD CC:          {psd_gw_cc:.4f}")

### 4. Save outputs

Visualize and compare reconstruction against ground truth using metrics (PSNR, SSIM) and visual inspection.

In [ ]:
# 4. Save outputs
print("\n[4/4] Saving results ...")

# Metrics
metrics = {
    "log10_A_gw_true": TRUE_LOG10_A_GW,
    "log10_A_gw_recovered": float(medians[0]),
    "log10_A_gw_std": float(stds[0]),
    "log10_A_gw_RE": re_values["log10_A_gw"],
    "log10_A_red_true": TRUE_LOG10_A_RED,
    "log10_A_red_recovered": float(medians[1]),
    "log10_A_red_std": float(stds[1]),
    "log10_A_red_RE": re_values["log10_A_red"],
    "gamma_red_true": TRUE_GAMMA_RED,
    "gamma_red_recovered": float(medians[2]),
    "gamma_red_std": float(stds[2]),
    "gamma_red_RE": re_values["gamma_red"],
    "mean_parameter_RE": mean_re,
    "red_noise_PSD_CC": psd_cc,
    "GW_PSD_CC": psd_gw_cc,
    "n_pulsars": N_PULSARS,
    "n_toa": N_TOA,
    "n_walkers": N_WALKERS,
    "n_steps": N_STEPS,
    "n_burn": N_BURN,
}
with open(os.path.join(RESULTS_DIR, "metrics.json"), "w") as f:
    json.dump(metrics, f, indent=2)
print("  → metrics.json saved")

# Ground truth & reconstruction arrays
gt_dict = {
    "true_params": true_params,
    "freqs": freqs,
    "psd_red_true": psd_true,
    "psd_gw_true": psd_gw_true,
    "residuals": [r for r in residuals_all],
    "hd_matrix": hd_matrix,
}
np.save(os.path.join(RESULTS_DIR, "ground_truth.npy"), gt_dict,
        allow_pickle=True)

recon_dict = {
    "recovered_params": medians,
    "param_stds": stds,
    "psd_red_recon": psd_recon,
    "psd_gw_recon": psd_gw_recon,
    "samples": samples,
}
np.save(os.path.join(RESULTS_DIR, "reconstruction.npy"), recon_dict,
        allow_pickle=True)
print("  → ground_truth.npy, reconstruction.npy saved")

# ── Visualization ─────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# (0,0) Trace plots
chain = sampler.get_chain()
for i, name in enumerate(param_names):
    ax = axes[0, 0] if i == 0 else (axes[0, 1] if i == 1 else axes[0, 2])
    for w in range(N_WALKERS):
        ax.plot(chain[:, w, i], alpha=0.3, lw=0.5)
    ax.axhline(true_params[i], color='r', lw=2, label='Truth')
    ax.axhline(medians[i], color='blue', ls='--', lw=1.5, label='Median')
    ax.set_xlabel('Step')
    ax.set_ylabel(name)
    ax.set_title(f'Trace: {name}')
    ax.legend(fontsize=8)

# (1,0) Red noise PSD
ax = axes[1, 0]
ax.loglog(freqs * 365.25 * 86400, psd_true, 'r-', lw=2, label='True red noise')
ax.loglog(freqs * 365.25 * 86400, psd_recon, 'b--', lw=2, label='Recovered')
ax.set_xlabel('Frequency (1/yr)')
ax.set_ylabel('PSD (s²/Hz)')
ax.set_title(f'Red Noise PSD (CC={psd_cc:.3f})')
ax.legend()
ax.grid(True, alpha=0.3)

# (1,1) GW PSD
ax = axes[1, 1]
ax.loglog(freqs * 365.25 * 86400, psd_gw_true, 'r-', lw=2, label='True GWB')
ax.loglog(freqs * 365.25 * 86400, psd_gw_recon, 'b--', lw=2, label='Recovered')
ax.set_xlabel('Frequency (1/yr)')
ax.set_ylabel('PSD (s²/Hz)')
ax.set_title(f'GW Background PSD (CC={psd_gw_cc:.3f})')
ax.legend()
ax.grid(True, alpha=0.3)

# (1,2) Corner-like: 2D posterior (A_gw vs A_red)
ax = axes[1, 2]
ax.scatter(samples[:, 0], samples[:, 1], s=1, alpha=0.2, c='steelblue')
ax.axvline(TRUE_LOG10_A_GW, color='r', lw=1.5, label='True')
ax.axhline(TRUE_LOG10_A_RED, color='r', lw=1.5)
ax.scatter([medians[0]], [medians[1]], c='blue', marker='x', s=100,
           zorder=5, label='Median')
ax.set_xlabel('log10_A_gw')
ax.set_ylabel('log10_A_red')
ax.set_title('Posterior: A_gw vs A_red')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

plt.tight_layout()
fig_path = os.path.join(RESULTS_DIR, "reconstruction_result.png")
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.close()
print(f"  → {fig_path} saved")

print("\n" + "=" * 60)
print("DONE — PTA Bayesian inference complete")
print(f"  Mean param RE = {mean_re:.4f}")
print(f"  Red PSD CC    = {psd_cc:.4f}")
print(f"  GW PSD CC     = {psd_gw_cc:.4f}")
print("=" * 60)

### Convergence Analysis

Tracking algorithm convergence is critical for understanding behavior. We plot:
- **Objective/loss** vs iteration number
- **Relative change** $\|\mathbf{x}^{(k+1)} - \mathbf{x}^{(k)}\| / \|\mathbf{x}^{(k)}\|$ to verify convergence
- **Reconstruction quality** (PSNR/SSIM) vs iteration if ground truth is available

A well-behaved algorithm should show monotonic decrease in the objective and eventual flattening.

In [ ]:
# === Convergence Analysis ===
# If the reconstruction code tracked iteration history, plot it here.
# Otherwise, this cell provides a template for convergence visualization.
import matplotlib.pyplot as plt
import numpy as np

# Example: if 'loss_history' or similar variable exists from reconstruction
try:
    # Try common variable names for convergence data
    conv_data = None
    for name in ['loss_history', 'losses', 'cost_history', 'residuals',
                  'objective', 'convergence', 'hist', 'history']:
        if name in dir():
            conv_data = eval(name)
            break
    
    if conv_data is not None and len(conv_data) > 1:
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        iterations = np.arange(1, len(conv_data) + 1)
        
        axes[0].semilogy(iterations, conv_data, 'b-', linewidth=1.5)
        axes[0].set_xlabel('Iteration', fontsize=12)
        axes[0].set_ylabel('Objective Value', fontsize=12)
        axes[0].set_title('Convergence Curve', fontsize=13)
        axes[0].grid(True, alpha=0.3)
        
        if len(conv_data) > 2:
            rel_change = np.abs(np.diff(conv_data)) / (np.abs(conv_data[:-1]) + 1e-12)
            axes[1].semilogy(iterations[1:], rel_change, 'r-', linewidth=1.5)
            axes[1].set_xlabel('Iteration', fontsize=12)
            axes[1].set_ylabel('Relative Change', fontsize=12)
            axes[1].set_title('Convergence Rate', fontsize=13)
            axes[1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        print(f"Final objective: {conv_data[-1]:.6e}")
        print(f"Total iterations: {len(conv_data)}")
    else:
        print("No convergence history found. The reconstruction may use a direct (non-iterative) method.")
except Exception as e:
    print(f"Convergence analysis skipped: {e}")
    print("Tip: Modify the reconstruction code to record loss at each iteration.")

### Error Map and Reconstruction Quality

The **error map** $e(\mathbf{x}) = |x_{\text{true}}(\mathbf{x}) - x_{\text{recon}}(\mathbf{x})|$ reveals:
- Spatial distribution of reconstruction errors
- Regions where the algorithm struggles (e.g., edges, fine details)
- Systematic biases in the reconstruction

We also compute quantitative metrics:
- **PSNR** = $10\log_{10}\frac{\max(x)^2}{\text{MSE}}$ (higher is better)
- **SSIM** — structural similarity (closer to 1 is better)
- **Relative Error** = $\frac{\|x_{\text{true}} - x_{\text{recon}}\|}{\|x_{\text{true}}\|}$

In [ ]:
# === Error Map Visualization ===
import matplotlib.pyplot as plt
import numpy as np

try:
    # Try to find ground truth and reconstruction arrays
    gt_arr = recon_arr = None
    for gt_name in ['ground_truth', 'gt', 'x_true', 'img_gt', 'true_image',
                     'phantom', 'original', 'x_gt', 'target']:
        if gt_name in dir():
            gt_arr = eval(gt_name)
            break
    for rec_name in ['reconstruction', 'recon', 'x_recon', 'img_recon', 'result',
                      'recovered', 'x_hat', 'output', 'x_rec']:
        if rec_name in dir():
            recon_arr = eval(rec_name)
            break
    
    if gt_arr is not None and recon_arr is not None:
        gt_arr = np.array(gt_arr, dtype=float)
        recon_arr = np.array(recon_arr, dtype=float)
        
        # Handle shape mismatch
        if gt_arr.shape != recon_arr.shape:
            min_shape = tuple(min(a, b) for a, b in zip(gt_arr.shape, recon_arr.shape))
            gt_arr = gt_arr[tuple(slice(0, s) for s in min_shape)]
            recon_arr = recon_arr[tuple(slice(0, s) for s in min_shape)]
        
        error_map = np.abs(gt_arr - recon_arr)
        
        if gt_arr.ndim == 2:  # 2D images
            fig, axes = plt.subplots(1, 4, figsize=(18, 4))
            
            im0 = axes[0].imshow(gt_arr, cmap='viridis')
            axes[0].set_title('Ground Truth', fontsize=12)
            plt.colorbar(im0, ax=axes[0], fraction=0.046)
            
            im1 = axes[1].imshow(recon_arr, cmap='viridis')
            axes[1].set_title('Reconstruction', fontsize=12)
            plt.colorbar(im1, ax=axes[1], fraction=0.046)
            
            im2 = axes[2].imshow(error_map, cmap='hot')
            axes[2].set_title('|Error Map|', fontsize=12)
            plt.colorbar(im2, ax=axes[2], fraction=0.046)
            
            # Histogram of errors
            axes[3].hist(error_map.ravel(), bins=50, color='steelblue', edgecolor='black', alpha=0.7)
            axes[3].set_xlabel('Absolute Error')
            axes[3].set_ylabel('Count')
            axes[3].set_title('Error Distribution', fontsize=12)
            
            for ax in axes[:3]:
                ax.axis('off')
        elif gt_arr.ndim == 1:  # 1D signals
            fig, axes = plt.subplots(2, 1, figsize=(12, 8))
            axes[0].plot(gt_arr, 'b-', label='Ground Truth', linewidth=1.5)
            axes[0].plot(recon_arr, 'r--', label='Reconstruction', linewidth=1.5)
            axes[0].legend(fontsize=11)
            axes[0].set_title('Signal Comparison', fontsize=13)
            axes[0].grid(True, alpha=0.3)
            
            axes[1].plot(error_map, 'k-', linewidth=1)
            axes[1].fill_between(range(len(error_map)), error_map, alpha=0.3, color='red')
            axes[1].set_title('Absolute Error', fontsize=13)
            axes[1].set_xlabel('Index')
            axes[1].grid(True, alpha=0.3)
        else:
            print(f"Data is {gt_arr.ndim}D — showing central slice")
            mid = gt_arr.shape[0] // 2
            fig, axes = plt.subplots(1, 3, figsize=(14, 4))
            axes[0].imshow(gt_arr[mid], cmap='viridis'); axes[0].set_title('GT (mid-slice)')
            axes[1].imshow(recon_arr[mid], cmap='viridis'); axes[1].set_title('Recon (mid-slice)')
            axes[2].imshow(error_map[mid], cmap='hot'); axes[2].set_title('Error (mid-slice)')
            for ax in axes: ax.axis('off')
        
        plt.tight_layout()
        plt.show()
        
        # Quantitative metrics
        mse = np.mean(error_map**2)
        rel_err = np.linalg.norm(error_map) / (np.linalg.norm(gt_arr) + 1e-12)
        data_range = gt_arr.max() - gt_arr.min()
        psnr = 10 * np.log10(data_range**2 / (mse + 1e-12)) if mse > 0 else float('inf')
        print(f"MSE:            {mse:.6e}")
        print(f"PSNR:           {psnr:.2f} dB")
        print(f"Relative Error: {rel_err:.6f}")
        print(f"Max Error:      {error_map.max():.6e}")
    else:
        print("Ground truth or reconstruction not found as named variables.")
        print("Modify this cell to reference your actual variable names.")
except Exception as e:
    print(f"Error map visualization skipped: {e}")

### Sensitivity Analysis

We analyze how reconstruction quality depends on key parameters:
- **Noise level**: How robust is the algorithm to increasing noise?
- **Regularization parameter** $\lambda$: What is the optimal trade-off?
- **Algorithm-specific parameters**: iterations, step size, etc.

This helps understand the algorithm's operating range and tune hyperparameters.

> **Note**: This analysis uses the variables defined earlier. If the reconstruction function
> is not available as a callable, this cell provides a template for manual parameter sweeps.

In [ ]:
# === Sensitivity / Parameter Sweep Analysis ===
import matplotlib.pyplot as plt
import numpy as np

print("Sensitivity Analysis Template")
print("=" * 50)
print()
print("To perform a full sensitivity analysis, uncomment and adapt the code below")
print("to match your specific reconstruction function and parameters.")
print()

# Template for noise sensitivity analysis:
# noise_levels = [0.01, 0.02, 0.05, 0.1, 0.2]
# psnr_values = []
# for sigma in noise_levels:
#     noisy_data = clean_data + sigma * np.random.randn(*clean_data.shape)
#     recon = run_reconstruction(noisy_data, ...)
#     psnr = compute_psnr(ground_truth, recon)
#     psnr_values.append(psnr)
#
# plt.figure(figsize=(8, 5))
# plt.plot(noise_levels, psnr_values, 'bo-', linewidth=2, markersize=8)
# plt.xlabel('Noise Level (sigma)', fontsize=12)
# plt.ylabel('PSNR (dB)', fontsize=12)
# plt.title('Reconstruction Quality vs Noise Level', fontsize=13)
# plt.grid(True, alpha=0.3)
# plt.show()

# Template for regularization parameter sweep:
# lambdas = np.logspace(-4, 1, 20)
# psnr_values = []
# for lam in lambdas:
#     recon = run_reconstruction(data, lambda_reg=lam, ...)
#     psnr = compute_psnr(ground_truth, recon)
#     psnr_values.append(psnr)
#
# plt.figure(figsize=(8, 5))
# plt.semilogx(lambdas, psnr_values, 'rs-', linewidth=2, markersize=8)
# plt.xlabel('Regularization Parameter (lambda)', fontsize=12)
# plt.ylabel('PSNR (dB)', fontsize=12)
# plt.title('L-curve: PSNR vs Regularization Strength', fontsize=13)
# plt.grid(True, alpha=0.3)
# plt.show()

print("Adapt the templates above to your specific forward model and reconstruction algorithm.")

## 6. Conclusion and Summary

This tutorial demonstrated the complete inverse problem pipeline for **enterprise_pta**:

1. **Problem Setup**: Spectroscopic measurements encode material properties in spectral features. Fitting extracts quantitative information.
2. **Forward Model**: Simulated measurements using the physical forward operator
3. **Inversion**: Applied the reconstruction algorithm to recover the unknown
4. **Evaluation**: Assessed reconstruction quality (PSNR=N/A (parameter estimation), SSIM=N/A)

### Key Takeaways
- The forward model maps unknown quantities to observable measurements
- Regularization is essential to stabilize the ill-posed inverse problem
- Quantitative metrics (PSNR, SSIM) and visual comparison validate the reconstruction

### Further Reading
- Paper: ENTERPRISE: Enhanced Numerical Toolbox Enabling a Robust PulsaR Inference SuitE
- Repository: https://github.com/nanograv/enterprise
